# cBAD $\rightarrow$ DINOv3 distillation $\rightarrow$ project PVTv2-B2

This notebook downloads all three splits of [`SSamDav/icdar-2019-competition-cbad`](https://huggingface.co/datasets/SSamDav/icdar-2019-competition-cbad), puts **images only** into one local Colab directory, and distils a DINOv3 ViT-B/16 teacher into the exact PVTv2-B2 architecture used by this project.

Student: `timm/pvt_v2_b2`, random initialization, no classifier. The exported state dict contains the exact hierarchical backbone used before the downstream U-Net decoder or the RT-DETR/Mask R-CNN feature pyramid. cBAD annotations are deliberately ignored.

> Select **Runtime → Change runtime type → GPU** before running the notebook. Data stay on the Colab VM; only experiment artifacts are written to Drive.

## 1. Install pinned dependencies

In [ ]:
%pip install -q "lightly-train[timm]==0.15.0" "timm==1.0.24" "datasets>=4.0,<5" pillow pandas matplotlib

import platform
import torch
import timm
import lightly_train

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("lightly-train:", getattr(lightly_train, "__version__", "unknown"))
assert torch.cuda.is_available(), "GPU is not enabled in this Colab runtime"
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 2. Mount Drive and configure the run

Lightly recommends batches of 128--2048 and 100--3000 epochs, with training up to 10000 epochs potentially beneficial for datasets below 100k images. Since cBAD contains only 3021 pages, this notebook uses 10000 epochs and an effective batch of 128. The physical micro-batch is selected from available VRAM and gradient accumulation supplies the remainder. The output directory is persistent, so an interrupted run can be resumed.

In [ ]:
from pathlib import Path
import math
from google.colab import drive

drive.mount("/content/drive")

DATASET_ID = "SSamDav/icdar-2019-competition-cbad"
DATA_DIR = Path("/content/cbad_images")  # local VM, not Drive
OUT_DIR = Path("/content/drive/MyDrive/thesis_pretraining/pvtv2_b2_cbad_dinov3")
FINAL_STATE_DICT = OUT_DIR / "pvtv2_b2_cbad_dinov3_backbone_state_dict.pt"

EPOCHS = 10_000
TARGET_EFFECTIVE_BATCH = 128
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
if VRAM_GIB >= 70:
    BATCH_SIZE = 128
elif VRAM_GIB >= 35:
    BATCH_SIZE = 64
elif VRAM_GIB >= 20:
    BATCH_SIZE = 32
else:
    BATCH_SIZE = 16
ACCUMULATE_GRAD_BATCHES = TARGET_EFFECTIVE_BATCH // BATCH_SIZE
assert BATCH_SIZE * ACCUMULATE_GRAD_BATCHES == TARGET_EFFECTIVE_BATCH
# LightlyTrain 0.15 scales LR from the physical batch and does not include
# Lightning accumulation. Compensate the default sqrt scaling so the final
# optimizer LR is exactly the one for an effective batch of 128.
ADAMW_BASE_LR = 5e-4
LIGHTLY_REFERENCE_BATCH = 1536
ADAMW_INPUT_LR = ADAMW_BASE_LR * math.sqrt(TARGET_EFFECTIVE_BATCH / BATCH_SIZE)
EXPECTED_SCALED_LR = ADAMW_BASE_LR * math.sqrt(
    TARGET_EFFECTIVE_BATCH / LIGHTLY_REFERENCE_BATCH
)
NUM_WORKERS = 4
IMAGE_SIZE = (224, 224)
SEED = 0
RESUME_INTERRUPTED = True
CHECKPOINT_EVERY_N_EPOCHS = 10   # full optimizer state; updated in Drive
EXPORT_EVERY_N_EPOCHS = 100     # native backbone export; updated in Drive

OUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print("Images (local):", DATA_DIR)
print("Artifacts (Drive):", OUT_DIR)
print("Epochs:", EPOCHS)
print("Physical batch:", BATCH_SIZE)
print("Gradient accumulation:", ACCUMULATE_GRAD_BATCHES)
print("Effective batch:", BATCH_SIZE * ACCUMULATE_GRAD_BATCHES)
print(f"Expected post-Lightly peak LR: {EXPECTED_SCALED_LR:.8f}")
print(f"Drive cadence: checkpoint every {CHECKPOINT_EVERY_N_EPOCHS} epochs; "
      f"backbone export every {EXPORT_EVERY_N_EPOCHS} epochs")

## 3. Stream all cBAD images from Hugging Face

The loader uses every available split and writes all pages into one flat directory. It requests the image column with decoding disabled and copies the original bytes; `alto`/XML is never written. Existing files make this cell restart-safe.

In [ ]:
from io import BytesIO
import re
from PIL import Image as PILImage
from datasets import Image as HFImage
from datasets import get_dataset_split_names, load_dataset

DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SUFFIX = {"JPEG": ".jpg", "PNG": ".png", "TIFF": ".tif", "WEBP": ".webp"}

def safe_name(value):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))
    return value.strip("._") or "page"

def original_image_bytes(payload):
    raw = payload.get("bytes")
    if raw is not None:
        return raw
    path = payload.get("path")
    if path and Path(path).is_file():
        return Path(path).read_bytes()
    raise RuntimeError(f"Hugging Face row has neither bytes nor a readable path: {payload}")

splits = get_dataset_split_names(DATASET_ID)
print("Hugging Face splits:", splits)
written = 0
for split in splits:
    stream = load_dataset(DATASET_ID, split=split, streaming=True)
    stream = stream.cast_column("image", HFImage(decode=False))
    split_count = 0
    for index, row in enumerate(stream):
        raw = original_image_bytes(row["image"])
        with PILImage.open(BytesIO(raw)) as probe:
            suffix = IMAGE_SUFFIX.get(probe.format, ".img")
            probe.verify()
        stem = safe_name(row.get("filename", f"{index:05d}"))
        target = DATA_DIR / f"{safe_name(split)}_{index:05d}_{stem}{suffix}"
        if not target.exists() or target.stat().st_size != len(raw):
            target.write_bytes(raw)
        split_count += 1
    written += split_count
    print(f"{split}: {split_count:,} images")

images = sorted(path for path in DATA_DIR.iterdir() if path.is_file())
print(f"Total rows streamed: {written:,}")
print(f"Files in the single image directory: {len(images):,}")
assert written == 3021, f"Expected 3,021 cBAD rows, got {written:,}"
assert len(images) == 3021, f"Expected 3,021 image files, got {len(images):,}"

In [ ]:
import matplotlib.pyplot as plt

sample = PILImage.open(images[0]).convert("RGB")
print("Example:", images[0].name, sample.size)
plt.figure(figsize=(10, 8))
plt.imshow(sample)
plt.axis("off")
plt.show()

## 4. Build and audit the exact project student

This is the exact `timm/pvt_v2_b2` hierarchical graph used by the project, initialized randomly. The hard guard stops immediately if Lightly would receive a truncated backbone.

In [ ]:
import torch
import timm

STUDENT_NAME = "pvt_v2_b2"
EXPECTED_PARAMS = 24_849_856
MIN_SAFE_PARAMS = 23_000_000

student = timm.create_model(
    STUDENT_NAME,
    pretrained=False,       # critical: cBAD student starts random
    num_classes=0,          # export backbone only
)

total_params = sum(parameter.numel() for parameter in student.parameters())
trainable_params = sum(
    parameter.numel() for parameter in student.parameters() if parameter.requires_grad
)
print(f"Student total parameters:     {total_params:,}")
print(f"Student trainable parameters: {trainable_params:,}")

if trainable_params < MIN_SAFE_PARAMS:
    raise RuntimeError(
        f"STOP: only {trainable_params:,} trainable parameters; "
        "the ViT backbone was probably truncated. Do not spend GPU time."
    )
assert trainable_params == EXPECTED_PARAMS, (
    f"Unexpected graph: {trainable_params:,} parameters instead of {EXPECTED_PARAMS:,}"
)

student.eval()
with torch.inference_mode():
    dense = student.forward_features(torch.zeros(1, 3, 224, 224))
print("Final dense feature map:", tuple(dense.shape))
assert tuple(dense.shape) == (1, 512, 7, 7)
student.train()

## 5. Distil DINOv3 ViT-B/16 into PVTv2-B2

`distillation` is LightlyTrain's current `distillationv3` alias. JSONL metrics are monitored in the background and a compact loss report is written every 100 epochs. A full resume checkpoint is refreshed in Drive every 10 epochs and the native backbone export is refreshed there every 100 epochs. Thus a Colab disconnect loses at most 10 epochs.

Run the one-epoch smoke test first if the runtime or dependency versions changed.

In [ ]:
# Set True for a separate one-epoch verification run; set False for the real run.
SMOKE_TEST = False
RUN_OUT = OUT_DIR / "smoke" if SMOKE_TEST else OUT_DIR
RUN_EPOCHS = 1 if SMOKE_TEST else EPOCHS
print("Run output:", RUN_OUT)
print("Epochs:", RUN_EPOCHS)

In [ ]:
import json
import threading
import time
import pandas as pd

LOSS_REPORT = RUN_OUT / "loss_every_100_epochs.csv"
monitor_stop = threading.Event()

def read_jsonl(path):
    records = []
    if not path.exists():
        return records
    for line in path.read_text(errors="replace").splitlines():
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass  # tolerate a line being written while we read it
    return records

def loss_snapshot(records, target_epoch):
    eligible = []
    for record in records:
        if "epoch" not in record:
            continue
        epoch = int(record["epoch"]) + 1  # Lightning epoch is zero-based
        losses = {
            key: value for key, value in record.items()
            if "loss" in key.lower() and isinstance(value, (int, float))
        }
        if epoch <= target_epoch and losses:
            eligible.append((epoch, record.get("step", -1), losses))
    return max(eligible, default=None, key=lambda item: (item[0], item[1]))

def monitor_losses():
    reported = {}
    metrics_path = RUN_OUT / "metrics.jsonl"
    while not monitor_stop.wait(30):
        records = read_jsonl(metrics_path)
        if not records:
            continue
        max_epoch = max((int(r.get("epoch", -1)) + 1 for r in records), default=0)
        for target in range(100, max_epoch + 1, 100):
            if target in reported:
                continue
            snapshot = loss_snapshot(records, target)
            if snapshot is None:
                continue
            actual_epoch, step, losses = snapshot
            row = {"target_epoch": target, "logged_epoch": actual_epoch, "step": step, **losses}
            reported[target] = row
            print(f"[100-epoch loss] {row}", flush=True)
            if target >= 600 and 500 in reported:
                common = [key for key in losses if key in reported[500] and "loss" in key.lower()]
                preferred = [key for key in common if "train" in key.lower()] or common
                if preferred:
                    key = sorted(preferred, key=len)[0]
                    if row[key] >= reported[500][key]:
                        print(
                            f"WARNING: {key} has not fallen relative to epoch 500: "
                            f"{reported[500][key]:.6g} -> {row[key]:.6g}", flush=True
                        )
        if reported:
            pd.DataFrame(reported.values()).sort_values("target_epoch").to_csv(LOSS_REPORT, index=False)

loss_thread = threading.Thread(target=monitor_losses, daemon=True)
loss_thread.start()
print("Loss monitor started; it will report epochs 100, 200, ...")

In [ ]:
import lightly_train

# Reprint immediately beside the expensive call: this is the object passed to LightlyTrain.
trainable_at_start = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"TRAINING START — trainable student parameters: {trainable_at_start:,}", flush=True)
if trainable_at_start < MIN_SAFE_PARAMS:
    raise RuntimeError("STOP: student has fewer than 23M trainable parameters")

try:
    lightly_train.pretrain(
        out=RUN_OUT,
        data=DATA_DIR,
        model=student,
        method="distillation",
        method_args={"teacher": "dinov3/vitb16"},
        batch_size=BATCH_SIZE,
        epochs=RUN_EPOCHS,
        num_workers=NUM_WORKERS,
        transform_args={"image_size": IMAGE_SIZE},
        precision="16-mixed",
        accelerator="gpu",
        devices=1,
        optim="adamw",
        optim_args={"lr": ADAMW_INPUT_LR},
        seed=SEED,
        resume_interrupted=RESUME_INTERRUPTED and not SMOKE_TEST,
        callbacks={
            "model_export": {
                "every_n_epochs": EXPORT_EVERY_N_EPOCHS if not SMOKE_TEST else 1
            },
            "model_checkpoint": {
                "every_n_epochs": CHECKPOINT_EVERY_N_EPOCHS if not SMOKE_TEST else 1,
                "save_last": True,
                "verbose": True,
            },
        },
        trainer_args={
            "log_every_n_steps": 10,
            "accumulate_grad_batches": ACCUMULATE_GRAD_BATCHES,
        },
    )
finally:
    monitor_stop.set()
    loss_thread.join(timeout=5)


## 6. Export and verify a strict backbone state dict

The explicit `torch_state_dict` export avoids framework wrappers. A freshly constructed project ViT must accept every tensor with `strict=True`; otherwise the artifact is rejected.

In [ ]:
from pathlib import Path
import shutil
import torch
import timm
import lightly_train

checkpoint = RUN_OUT / "checkpoints" / "last.ckpt"
assert checkpoint.is_file(), f"Missing LightlyTrain checkpoint: {checkpoint}"
export_path = (RUN_OUT / "pvtv2_b2_cbad_dinov3_backbone_state_dict.pt"
               if SMOKE_TEST else FINAL_STATE_DICT)

lightly_train.export(
    out=export_path,
    checkpoint=checkpoint,
    part="model",
    format="torch_state_dict",
    overwrite=True,
)

state_dict = torch.load(export_path, map_location="cpu", weights_only=True)
fresh = timm.create_model(
    STUDENT_NAME, pretrained=False, num_classes=0
)
incompatible = fresh.load_state_dict(state_dict, strict=True)
loaded_params = sum(tensor.numel() for tensor in state_dict.values())
print("Strict load result:", incompatible)
print(f"Exported tensors cover {loaded_params:,} parameters")
assert loaded_params == EXPECTED_PARAMS
print("Verified backbone in Drive:", export_path)

# Keep the compact loss report next to the final weights.
records = read_jsonl(RUN_OUT / "metrics.jsonl")
rows = []
for target in range(100, RUN_EPOCHS + 1, 100):
    snapshot = loss_snapshot(records, target)
    if snapshot is not None:
        epoch, step, losses = snapshot
        rows.append({"target_epoch": target, "logged_epoch": epoch, "step": step, **losses})
if rows:
    pd.DataFrame(rows).to_csv(LOSS_REPORT, index=False)
    display(pd.DataFrame(rows))

## 7. Downstream loading contract

Load this file into the same `pvt_v2_b2` backbone **before** attaching a downstream decoder or feature pyramid. Use `strict=True`; accepting partial keys would invalidate the comparison.

In [ ]:
import torch
import timm

encoder = timm.create_model(
    "pvt_v2_b2",
    pretrained=False,
    num_classes=0,
)
weights = torch.load(FINAL_STATE_DICT, map_location="cpu", weights_only=True)
encoder.load_state_dict(weights, strict=True)
print("Ready for a downstream decoder/FPN; parameters:", sum(p.numel() for p in encoder.parameters()))